# 01 — Data Loading and Quality Checks

Stage 02. Loads the raw UCI Beijing PM2.5 dataset reproducibly via `src.data`, verifies timestamps, dtypes, duplicates, missingness and invalid values. No cleaning beyond renaming/typing is applied; the raw file is never modified.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data import RAW_PATH, RAW_SHA256, file_sha256, load_raw, validate, missing_runs

pd.set_option('display.width', 120)

## 1. Provenance and integrity
Source: UCI Machine Learning Repository, *Beijing PM2.5 Data* (Chen, 2015, doi:10.24432/C5JS49, CC BY 4.0). PM2.5 from the US Embassy in Beijing; weather from Beijing Capital International Airport.

In [2]:
sha = file_sha256(RAW_PATH)
print(RAW_PATH.relative_to(ROOT))
print('SHA-256 matches record:', sha == RAW_SHA256)

data/raw/dataset.csv
SHA-256 matches record: True


## 2. Raw file as stored

In [3]:
raw = pd.read_csv(RAW_PATH, na_values=['NA'], keep_default_na=False)
print(raw.shape)
raw.head()

(43824, 13)


,No,year,month,day,hour,pm2.5,DEWP,TEMP,PRES,cbwd,Iws,Is,Ir
0,1,2010,1,1,0,NaN,-21,-11.0,1021.0,NW,1.79,0,0
1,2,2010,1,1,1,NaN,-21,-12.0,1020.0,NW,4.92,0,0
2,3,2010,1,1,2,NaN,-21,-11.0,1019.0,NW,6.71,0,0
3,4,2010,1,1,3,NaN,-21,-14.0,1019.0,NW,9.84,0,0
4,5,2010,1,1,4,NaN,-20,-12.0,1018.0,NW,12.97,0,0


In [4]:
raw.dtypes.to_frame('dtype').T

,No,year,month,day,hour,pm2.5,DEWP,TEMP,PRES,cbwd,Iws,Is,Ir
dtype,int64,int64,int64,int64,int64,float64,int64,float64,float64,str,float64,int64,int64


## 3. Load and validate
`load_raw` builds the timestamp from year/month/day/hour, renames columns and types `wind_dir` as categorical. `validate` raises on any hard-invariant failure.

In [5]:
df = load_raw()
print('validate:', validate(df))
df.head()

validate: True


,pm25,dewp,temp,pres,wind_dir,wind_cum,snow_cum,rain_cum
timestamp,,,,,,,,
2010-01-01 00:00:00,NaN,-21,-11.0,1021.0,NW,1.79,0,0
2010-01-01 01:00:00,NaN,-21,-12.0,1020.0,NW,4.92,0,0
2010-01-01 02:00:00,NaN,-21,-11.0,1019.0,NW,6.71,0,0
2010-01-01 03:00:00,NaN,-21,-14.0,1019.0,NW,9.84,0,0
2010-01-01 04:00:00,NaN,-20,-12.0,1018.0,NW,12.97,0,0


## 4. Timestamps and frequency

In [6]:
idx = df.index
pd.Series({
    'start': idx.min(), 'end': idx.max(), 'rows': len(idx),
    'expected hourly rows': len(pd.date_range(idx.min(), idx.max(), freq='h')),
    'duplicate timestamps': idx.duplicated().sum(),
    'monotonic increasing': idx.is_monotonic_increasing,
    'step sizes': idx.to_series().diff().dropna().value_counts().to_dict(),
    'raw No is 1..N': (raw['No'] == range(1, len(raw) + 1)).all(),
    'duplicate rows (all columns)': raw.drop(columns='No').duplicated().sum(),
})

start                                2010-01-01 00:00:00
end                                  2014-12-31 23:00:00
rows                                               43824
expected hourly rows                               43824
duplicate timestamps                                   0
monotonic increasing                                True
step sizes                      {0 days 01:00:00: 43823}
raw No is 1..N                                      True
duplicate rows (all columns)                           0
dtype: object

## 5. Missingness

In [7]:
df.isna().sum().to_frame('missing').assign(pct=lambda t: (100 * t['missing'] / len(df)).round(2))

,missing,pct
pm25,2067,4.72
dewp,0,0.00
temp,0,0.00
pres,0,0.00
wind_dir,0,0.00
wind_cum,0,0.00
snow_cum,0,0.00
rain_cum,0,0.00


In [8]:
runs = missing_runs(df['pm25'])
bins = pd.cut(runs['hours'], [0, 1, 3, 6, 24, 72, 1000],
              labels=['1 h', '2-3 h', '4-6 h', '7-24 h', '25-72 h', '>72 h'])
(runs.groupby(bins, observed=False)['hours']
     .agg(gaps='size', missing_hours='sum'))

,gaps,missing_hours
hours,,
1 h,112,112
2-3 h,31,64
4-6 h,15,74
7-24 h,36,476
25-72 h,13,645
>72 h,7,696


In [9]:
runs.nlargest(8, 'hours')

,start,end,hours
20,2010-09-21 05:00:00,2010-09-27 15:00:00,155
143,2012-12-23 06:00:00,2012-12-28 12:00:00,127
81,2011-10-03 13:00:00,2011-10-07 15:00:00,99
30,2011-03-17 21:00:00,2011-03-21 15:00:00,91
21,2010-09-27 17:00:00,2010-09-30 20:00:00,76
92,2012-01-14 10:00:00,2012-01-17 11:00:00,74
129,2012-08-18 11:00:00,2012-08-21 12:00:00,74
13,2010-06-04 12:00:00,2010-06-07 11:00:00,72


In [10]:
df['pm25'].isna().groupby(df.index.year).mean().mul(100).round(1).rename('% missing by year').to_frame().T

timestamp,2010,2011,2012,2013,2014
% missing by year,7.6,8.3,5.6,0.9,1.1


## 6. Value ranges and suspicious values

In [11]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
pm25,41757.0,98.61,92.05,0.00,29.00,72.00,137.00,994.0
dewp,43824.0,1.82,14.43,-40.00,-10.00,2.00,15.00,28.0
temp,43824.0,12.45,12.20,-19.00,2.00,14.00,23.00,42.0
pres,43824.0,1016.45,10.27,991.00,1008.00,1016.00,1025.00,1046.0
wind_cum,43824.0,23.89,50.01,0.45,1.79,5.37,21.91,585.6
snow_cum,43824.0,0.05,0.76,0.00,0.00,0.00,0.00,27.0
rain_cum,43824.0,0.19,1.42,0.00,0.00,0.00,0.00,36.0


In [12]:
df['wind_dir'].value_counts()

wind_dir
SE    15290
NW    14150
cv     9387
NE     4997
Name: count, dtype: int64

In [13]:
pm = df['pm25']
print('PM2.5 == 0:', pm.eq(0).sum(), list(pm.index[pm.eq(0)]))
print('PM2.5 > 500:', pm.gt(500).sum())
pm.nlargest(5)

PM2.5 == 0: 2 [Timestamp('2012-09-28 10:00:00'), Timestamp('2012-09-28 15:00:00')]
PM2.5 > 500: 125


timestamp
2012-01-23 01:00:00    994.0
2010-02-14 01:00:00    980.0
2012-01-23 02:00:00    972.0
2013-01-12 20:00:00    886.0
2013-01-12 22:00:00    858.0
Name: pm25, dtype: float64

The two highest values fall on Chinese New Year's Eve/Day (2010-02-14, 2012-01-23), consistent with fireworks episodes; Jan 2013 was a widely reported severe haze period. Extremes are therefore treated as plausible and kept.

In [14]:
print('dew point > temperature:')
df.loc[df['dewp'] > df['temp'], ['dewp', 'temp', 'pres', 'pm25']]

dew point > temperature:


,dewp,temp,pres,pm25
timestamp,,,,
2010-10-08 07:00:00,13,12.0,1016.0,349.0


In [15]:
for c in ['temp', 'pres']:
    nonint = df[c][df[c] != df[c].round()]
    print(c, 'non-integer values:', nonint.round(2).to_dict())

temp non-integer values: {Timestamp('2014-11-03 19:00:00'): 14.67, Timestamp('2014-11-03 20:00:00'): 9.33}
pres non-integer values: {Timestamp('2010-11-12 17:00:00'): 1019.5, Timestamp('2010-11-13 18:00:00'): 1029.67, Timestamp('2010-11-13 19:00:00'): 1032.33, Timestamp('2013-02-21 13:00:00'): 1027.5}


## 7. Cumulative variables
UCI describes `Iws`, `Is`, `Ir` as *cumulated*. Check how they accumulate.

In [16]:
same_dir = df['wind_dir'].eq(df['wind_dir'].shift())
step = df['wind_cum'].diff()
print('wind_cum increases whenever direction is unchanged:', (step[same_dir] > 0).all())
print('wind_cum on direction change (most common values):', df['wind_cum'][~same_dir].value_counts().head(3).to_dict())
for c in ['snow_cum', 'rain_cum']:
    s = df[c]; d = s.diff()
    irregular = ((s > 0) & (s.shift() > 0) & (d != 1)).sum()
    print(f'{c}: hours > 0 = {(s > 0).sum()}, consecutive steps not +1 = {irregular}')
print('snow and rain both > 0:', ((df['snow_cum'] > 0) & (df['rain_cum'] > 0)).sum())

wind_cum increases whenever direction is unchanged: True
wind_cum on direction change (most common values): {0.89: 6266, 1.79: 4756, 3.13: 1920}
snow_cum: hours > 0 = 368, consecutive steps not +1 = 6
rain_cum: hours > 0 = 1808, consecutive steps not +1 = 38
snow and rain both > 0: 0


## 8. Findings and decisions
- Index is a complete, unique, sorted hourly range 2010-01-01 00:00 → 2014-12-31 23:00 (43,824 rows); timestamps assumed Beijing local time (UTC+8, no DST).
- Only `pm25` is missing (2,067 h, 4.7%) in 214 gaps; most gaps are 1–3 h but a few last 3–6 days. Missingness is higher in 2010–2012. **NaNs are kept; handling is decided in Stage 04.**
- Extreme PM2.5 values (max 994) coincide with known pollution/fireworks events → kept. Two exact zeros are kept but flagged.
- One row has dew point 1 °C above temperature (likely integer rounding near saturation) → kept, flagged.
- Two non-integer temperatures and four non-integer pressures (values like x.33, x.67) suggest prior interpolation in the source → kept.
- `wind_cum` accumulates while direction is unchanged and resets on change; `snow_cum`/`rain_cum` count consecutive precipitation hours with a few skipped increments. These are raw cumulative signals and will be decoded deliberately in feature engineering (Stage 05).